# Notebook 03 — Frequently Bought Together Analysis
**Author:** Ishara  
**Role:** Spark SQL Analysis  
**Task:** Load cleaned data from HDFS → Find product pairs bought in the same order → Rank by frequency → Save results

---
> **Prerequisite:** `01_data_cleaning.ipynb` must have been run first.

**Technique:** Market Basket Analysis using a self-join.  
We join the transactions table with itself on `Invoice`, pairing any two different products that appear in the same order, then count how often each pair appears.

## 1. Load Packages

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.makedirs("output/frequent_pairs", exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

print("Packages loaded.")

## 2. Start Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("FrequentlyBoughtTogether") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 3. Load Cleaned Dataset from Hadoop HDFS

In [ ]:
# Load cleaned Parquet — saved by Notebook 01
df = spark.read.parquet("hdfs:///retail/cleaned/")

print(f"Loaded {df.count():,} records from HDFS.")
df.show(3, truncate=True)

## 4. Prepare Data for Pair Analysis

We keep only the columns we need: `Invoice`, `StockCode`, `Description`.  
We also deduplicate so each product appears only once per invoice (avoids counting quantity multiples as separate items).

In [ ]:
# Keep one row per unique Invoice + StockCode combination
basket = df.select("Invoice", "StockCode", "Description").distinct()

# Only keep invoices with 2 or more products (required for pair analysis)
invoice_sizes = basket.groupBy("Invoice").count().filter(F.col("count") >= 2)
basket = basket.join(invoice_sizes.select("Invoice"), on="Invoice", how="inner")

print(f"Basket rows (unique Invoice-Product pairs): {basket.count():,}")
print(f"Invoices with 2+ products: {invoice_sizes.count():,}")

## 5. Register Temp View for Spark SQL

In [ ]:
# Register as SQL table so we can use Spark SQL joins
basket.createOrReplaceTempView("basket")
print("Temp view 'basket' registered.")

## 6. Frequently Bought Together — Spark SQL Self-Join

**Design Pattern:** JOIN + AGGREGATION  
- We join `basket` with itself on `Invoice` to pair up products in the same order  
- `a.StockCode < b.StockCode` ensures each pair is counted only once (no duplicates like A+B and B+A)  
- `GROUP BY` + `COUNT` ranks pairs by how often they appear together

In [ ]:
# Spark SQL self-join to find product pairs in the same invoice
pair_query = """
    SELECT
        a.StockCode      AS ProductA_Code,
        a.Description    AS ProductA_Name,
        b.StockCode      AS ProductB_Code,
        b.Description    AS ProductB_Name,
        COUNT(*)         AS PairCount
    FROM basket a
    JOIN basket b
        ON  a.Invoice    = b.Invoice
        AND a.StockCode  < b.StockCode
    GROUP BY
        a.StockCode, a.Description,
        b.StockCode, b.Description
    HAVING COUNT(*) >= 10
    ORDER BY PairCount DESC
"""

pairs = spark.sql(pair_query)
print(f"Product pairs found (min 10 co-occurrences): {pairs.count():,}")
pairs.show(10, truncate=40)

## 7. Top 20 Most Frequently Bought Together Pairs

In [ ]:
# Get top 20 pairs
top_pairs = pairs.limit(20).toPandas()

# Create a readable label for each pair
top_pairs["PairLabel"] = (
    top_pairs["ProductA_Name"].str[:22].str.strip() +
    " + " +
    top_pairs["ProductB_Name"].str[:22].str.strip()
)

print("Top 20 product pairs:")
print(top_pairs[["ProductA_Name", "ProductB_Name", "PairCount"]].to_string(index=False))

## 8. Calculate Support, Confidence & Lift (Association Metrics)

These metrics make the analysis more meaningful for the report:
- **Support** = how often the pair appears out of all invoices
- **Confidence** = P(Product B | Product A purchased)
- **Lift** = how much more likely than random chance

In [ ]:
# Total number of invoices
total_invoices = df.select("Invoice").distinct().count()

# Count how often each individual product appears in an invoice
product_freq_query = """
    SELECT StockCode, COUNT(DISTINCT Invoice) AS Freq
    FROM basket
    GROUP BY StockCode
"""
product_freq = spark.sql(product_freq_query)
product_freq.createOrReplaceTempView("product_freq")

# Enrich pairs with support and confidence
metrics_query = f"""
    SELECT
        p.ProductA_Code,
        p.ProductA_Name,
        p.ProductB_Code,
        p.ProductB_Name,
        p.PairCount,
        ROUND(p.PairCount / {total_invoices}, 4)                   AS Support,
        ROUND(p.PairCount / fa.Freq, 4)                             AS Confidence_AtoB,
        ROUND((p.PairCount / {total_invoices}) /
              ((fa.Freq / {total_invoices}) * (fb.Freq / {total_invoices})), 2) AS Lift
    FROM (
        SELECT * FROM basket_pairs
    ) p
    JOIN product_freq fa ON p.ProductA_Code = fa.StockCode
    JOIN product_freq fb ON p.ProductB_Code = fb.StockCode
    ORDER BY Lift DESC
"""

# Register pairs as temp view for the metrics query
pairs.createOrReplaceTempView("basket_pairs")
pairs_metrics = spark.sql(metrics_query)

print("Top pairs with association metrics:")
pairs_metrics.show(10, truncate=35)

## 9. Visualise — Top 15 Pairs Chart

In [ ]:
top15 = top_pairs.head(15)

fig, ax = plt.subplots(figsize=(11, 7))
colors = sns.color_palette("Blues_r", len(top15))

bars = ax.barh(top15["PairLabel"][::-1], top15["PairCount"][::-1], color=colors)

ax.set_xlabel("Times Bought Together", fontsize=12)
ax.set_title("Top 15 Frequently Bought Together Product Pairs", fontsize=14, fontweight="bold", pad=15)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 2, bar.get_y() + bar.get_height()/2,
            f"{int(w):,}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("output/frequent_pairs/chart_frequent_pairs.png", bbox_inches="tight")
plt.show()
print("Chart saved.")

## 10. Visualise — Pair Count Distribution

In [ ]:
# Distribution of pair frequencies
all_pairs_pd = pairs.select("PairCount").filter(F.col("PairCount") <= 200).toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_pairs_pd["PairCount"], bins=40, color="#2E75B6", edgecolor="white", alpha=0.85)
ax.set_xlabel("Times Bought Together", fontsize=12)
ax.set_ylabel("Number of Pairs", fontsize=12)
ax.set_title("Distribution of Product Pair Co-occurrence Counts", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("output/frequent_pairs/chart_pair_distribution.png", bbox_inches="tight")
plt.show()
print("Distribution chart saved.")

## 11. Save Results — HDFS & Local

In [ ]:
# ── Save top 500 pairs to HDFS as Parquet ────────────────────────────────────
pairs.limit(500).write.parquet(
    "hdfs:///retail/output/frequent_pairs/",
    mode="overwrite"
)
print("Results saved to HDFS: hdfs:///retail/output/frequent_pairs/")

# ── Save top 100 pairs locally as CSV (for GitHub / report) ──────────────────
top_pairs_save = pairs.limit(100).toPandas()
top_pairs_save.to_csv("output/frequent_pairs/top100_product_pairs.csv", index=False)
print("CSV saved locally:      output/frequent_pairs/top100_product_pairs.csv")

# ── Save metrics result locally ───────────────────────────────────────────────
pairs_metrics.limit(100).toPandas().to_csv(
    "output/frequent_pairs/top100_pairs_with_metrics.csv", index=False
)
print("Metrics CSV saved:      output/frequent_pairs/top100_pairs_with_metrics.csv")

## 12. Insights Summary

In [ ]:
total_pairs = pairs.count()
top1 = top_pairs.iloc[0]

print("=" * 55)
print(f"  Total valid pairs found   : {total_pairs:>10,}")
print(f"  Most common pair          : #{1}")
print(f"    Product A : {top1['ProductA_Name'][:40]}")
print(f"    Product B : {top1['ProductB_Name'][:40]}")
print(f"    Bought together: {int(top1['PairCount']):,} times")
print("=" * 55)

**Interpretation:**  
The frequently bought together analysis reveals strong product affinity patterns — clusters of small decorative or utility items that customers consistently purchase in the same order. These pairs have high lift values, meaning they are bought together far more often than would occur by chance. Retailers can use these insights to:
- Place frequently paired products near each other on the website/store
- Create bundle promotions ("Buy A, get B at a discount")
- Power "customers also bought" recommendation widgets

---
## Summary

| Step | Action | Design Pattern |
|------|--------|----------------|
| 1 | Load cleaned data from HDFS | Data Loading |
| 2 | Deduplicate basket (1 row per invoice-product) | Filtering |
| 3 | Self-join basket on Invoice + StockCode < StockCode | JOIN |
| 4 | GROUP BY pair, COUNT occurrences | Aggregation |
| 5 | HAVING COUNT >= 10, ORDER BY DESC | Filtering + Sorting |
| 6 | Calculate Support, Confidence, Lift | Transformation |
| 7 | Visualise top 15 pairs as bar chart | Output |
| 8 | Save to HDFS (Parquet) + local (CSV) | Output |

**Next:** Run `04_recommendations.ipynb`